# ROT ベンチマーク — ローカル推論モデルで `<think>` を測る

自己記述性の水準を10段に振ったデータに対して課題を解かせ、**成果あたりのトークン消費**と
**中間推論（`<think>`）のテキストそのもの**を記録します。

API 経由のモデル（gpt-4o-mini / gpt-4.1-mini / gpt-5.4）は `reasoning_tokens` が 0 で返り、
中間推論の長さを分離できませんでした。推論モデルを自前で回すと、思考が生のテキストで取れます。

既定のモデルは [`allenai/Olmo-3-7B-Think`](https://huggingface.co/allenai/Olmo-3-7B-Think) です。
重み・学習コード・学習データが揃って公開されており、OSI の
[Open Source AI Definition](https://opensource.org/ai/open-source-ai-definition) を満たす
数少ない系統です（Qwen 系などは重みのみ公開で、この定義は満たしません）。

## 実行前に

**ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: A100 GPU** を選んでください。

* 7B は A100 40GB でも動きます。32B を選ぶ場合は 80GB が要ります。
* **タブを開いたままにしてください。** 閉じるとランタイムが切れます。

## 所要時間の目安

| 反復 | 目安 |
| --- | --- |
| `REPEATS=1`（既定） | **約2.5時間**（100実行。うち6セルが試行上限まで回る想定） |
| `REPEATS=2` | 約5時間 |
| `REPEATS=5` | 約12.6時間。ブラウザ経路では現実的でありません |

1試行あたりの生成が平均 10,400 トークン、A100 40GB で約 70 tok/s という実測からの外挿です。
**上限10試行まで完走した実測はまだ無い**ので、下振れの可能性があります。

## 途中で落ちたとき

本実行は**1試行ごとに途中経過を書き足します**。ランタイムが切れても、
セル2（取得と起動）とセル4（本実行）をもう一度実行すれば、**済んだ分を飛ばして続きから回ります**。

途中経過の置き場所は設定の指紋で決まるので、設定やコードを変えると別のランとして扱われ、混ざりません。

## 記録されるもの

* `attempt_log[].thinking` — **思考のテキストそのもの**
* `thinking_chars` — その文字数。**集計の「思考字数」列に出ます**（`CoT` 列は usage 由来の `reasoning_tokens` で、パーサ無しの vLLM では `n/a` になります）
* `output_capped_attempts` — 生成上限に達した試行数。**到達しても集計から除外しません**
* `fingerprint` — 投げたデータ・タスク・プロンプト・サンプリング設定のハッシュと、clone したコミット


## 1. 設定

変えるならここだけです。既定は参照点のランと同じ設定にしてあります。


In [ ]:
MODEL   = 'allenai/Olmo-3-7B-Think'   # 32B にするなら A100 80GB が要る
REPEATS = '1'                          # まずは 1。ばらつきを見るなら増やす
TASKS   = 'task_04,task_06'            # 逃げ道のある課題と、無い課題
SUITE   = 'v3_levels'

MAX_ATTEMPTS      = '10'      # 1タスクあたりの試行上限（API 経由の測定と揃えてある）
MAX_OUTPUT_TOKENS = '32768'   # 1リクエストの生成上限。短く切ると切った位置が測定値を決める
MAX_MODEL_LEN     = 65536     # 思考が発散しても文脈長で落ちないよう広めに取る

TEMPERATURE, TOP_P, SEED = '1.0', '1.0', '20260820'

# 実行経路の覚え書き。結果に残り、ラン記録（results/reference/runs/）に出る。
# 何で回したかは結果から機械的には分からないので、ここに書いておく。
RUN_ROUTE = ('Colab ブラウザ経路 / colab/run_local_model.ipynb / '
             'vllm serve --max-model-len ' + str(MAX_MODEL_LEN) +
             ' --gpu-memory-utilization 0.90（reasoning-parser なし）')
REPO = 'https://github.com/beachcities/RoT.git'


## 2. 取得と起動

リポジトリを clone し、依存を入れて vLLM を立ち上げます。モデルの読み込みまで含めて5〜10分。

`git clone` にしてあるのは、**どのコミットで回したかが結果の指紋に残る**ためです。
手元のファイルを上げる形だと、リポジトリの版と手元の版がずれます。


In [ ]:
import subprocess, os, sys, time, urllib.request, shutil

if not os.path.isdir('/content/RoT'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, '/content/RoT'], check=True)
BENCH = '/content/RoT/benchmark'
print('commit:', subprocess.run(['git', '-C', '/content/RoT', 'rev-parse', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())

# Colab のイメージは torch(CUDA 13.0) と torchaudio(12.8) が食い違っており、
# vLLM の import 経路で落ちる。テキスト推論に torchaudio は要らないので外す。
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchaudio'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai', 'python-dotenv'])
try:
    import vllm
    print('vllm', vllm.__version__)
except ImportError:
    print('vllm を入れます（数分）')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

with open(BENCH + '/.env', 'w') as f:
    f.write('API_KEY=dummy' + chr(10) + 'BASE_URL=http://127.0.0.1:8000/v1' + chr(10))

# reasoning-parser は付けない。使えるかはモデルのトークナイザ次第で、
# 付けなくても </think> は本文に残り、ランナー側が切り出す。
log = open('/content/vllm.log', 'w')
server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--port', '8000',
     '--max-model-len', str(MAX_MODEL_LEN), '--gpu-memory-utilization', '0.90'],
    stdout=log, stderr=subprocess.STDOUT)

for i in range(120):
    time.sleep(10)
    if server.poll() is not None:
        print('サーバが落ちました')
        print(open('/content/vllm.log').read()[-3000:])
        break
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3)
        print('READY after', (i + 1) * 10, 's')
        break
    except Exception:
        pass
else:
    print('まだ読み込み中')
    print(open('/content/vllm.log').read()[-2000:])

gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                           '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
print('GPU:', gpu_name)


## 3. 疎通の確認

本実行の前に1セルだけ回して、`<think>` が取れているかを見ます。1分程度。


In [ ]:
env = dict(os.environ, PYTHONIOENCODING='utf-8',
           MAX_ATTEMPTS=MAX_ATTEMPTS, REPEATS='1', SUITE=SUITE, PROMPT='p1_baseline',
           TEMPERATURE=TEMPERATURE, TOP_P=TOP_P, SEED=SEED,
           MAX_OUTPUT_TOKENS=MAX_OUTPUT_TOKENS, REQUEST_TIMEOUT='1800', MAX_RETRIES='2',
           RUN_ROUTE=RUN_ROUTE + ' / GPU: ' + gpu_name)

r = subprocess.run([sys.executable, 'run_benchmark.py', '--models', MODEL,
                    '--conditions', 'l6_codes_doc', '--tasks', 'task_06', '--no-save'],
                   cwd=BENCH, capture_output=True, text=True, env=env)
print(r.stdout[-1500:] or r.stderr[-1500:])


## 4. 本実行

進捗は `running:` の行で追えます。**2.5時間ほどかかります。**


In [ ]:
t0 = time.time()
env = dict(env, REPEATS=REPEATS)
proc = subprocess.Popen([sys.executable, '-u', 'run_benchmark.py',
                         '--models', MODEL, '--tasks', TASKS],
                        cwd=BENCH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=env)
for line in proc.stdout:
    print(line, end='', flush=True)
print(chr(10) + '== 所要 ' + str(round((time.time() - t0) / 60)) + '分 / rc=' + str(proc.wait()) + ' ==')


## 5. 結果の回収

`/content/` に結果JSONを置きます。左のファイルペインからダウンロードしてください。
思考のテキストを含むので数MBになります。


In [ ]:
import glob, json
f = sorted(glob.glob(BENCH + '/results/run_*.json'))[-1]
dest = '/content/' + os.path.basename(f)
shutil.copy(f, dest)
d = json.load(open(f, encoding='utf-8'))
print('保存先:', dest, '(%.1f MB)' % (os.path.getsize(f) / 1e6))
print('コミット:', d['fingerprint']['git']['commit'])
print('入力の指紋:', d['fingerprint']['inputs'])
print('生成上限:', d['fingerprint']['settings']['sampling_requested'].get('max_tokens'))
print('生成上限に達した試行:', sum(x.get('output_capped_attempts') or 0 for x in d['results']))
chars = [x['thinking_chars'] for x in d['results'] if x['thinking_chars']]
print('思考の文字数: 件数', len(chars), '/ 最大', max(chars) if chars else None)
